In [1]:
import os
import logging

# 1. Force C++ backend to only show FATAL errors
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

# 2. Silence Python's absl logging module before importing TensorFlow
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

# 3. Silence Python's standard logging for TensorFlow
logging.getLogger('tensorflow').setLevel(logging.FATAL)

import tensorflow as tf

In [2]:
import sys

# --- CONFIGURATION ---
REPO_NAME = "RefraScan"
# Make sure your username and repo name are spelled EXACTLY as they appear on GitHub
GITHUB_USER = "KyziaPi" 
BRANCH_NAME = "DenseNet121-Train"  
# ---------------------

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_PATH = os.path.join("/kaggle/working", REPO_NAME)

if not os.path.exists(REPO_NAME):
    print(f"Cloning branch '{BRANCH_NAME}' from {REPO_NAME}...")
    # env={"GIT_TERMINAL_PROMPT": "0"} tells Git to instantly fail instead of hanging if it can't find the repo
    !env GIT_TERMINAL_PROMPT=0 git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"{REPO_NAME} already exists. Switching branch and pulling latest updates...")
    !cd {REPO_NAME} && env GIT_TERMINAL_PROMPT=0 git fetch --all && git checkout {BRANCH_NAME} && git pull origin {BRANCH_NAME}

# Add the repository root to sys.path so Python can find the 'src' package
if os.path.exists(REPO_PATH):
    if REPO_PATH not in sys.path:
        sys.path.append(REPO_PATH)
    
    # Import your modular components
    import math
    import pandas as pd
    from tensorflow.keras.applications.densenet import preprocess_input

    try:
        from src.preprocessing import load_and_clean_data
        from src.cross_validation import run_cross_validation
        print("🚀 Success! Custom modules imported smoothly.")
    except ModuleNotFoundError as e:
        print(f"❌ Still failing. Current sys.path contains: {sys.path}")
        raise e

    print(f"Environment configured successfully! Working on branch: {BRANCH_NAME}")
else:
    print("❌ Error: Repository failed to clone. Double check your GITHUB_USER name and ensure the branch 'preprocessing_test' actually exists on GitHub.")

RefraScan already exists. Switching branch and pulling latest updates...
Fetching origin
Already on 'DenseNet121-Train'
Your branch is up to date with 'origin/DenseNet121-Train'.
From https://github.com/KyziaPi/RefraScan
 * branch            DenseNet121-Train -> FETCH_HEAD
Already up to date.
🚀 Success! Custom modules imported smoothly.
Environment configured successfully! Working on branch: DenseNet121-Train


In [3]:
# 1. Configure Paths
DATASET_DIR = '/kaggle/input/datasets/yerikaelainegueco/fundus-images-with-refractive-values' 
CSV_PATH = os.path.join(DATASET_DIR, 'RefraScan_dataset.csv')
IMG_DIR = os.path.join(DATASET_DIR, 'FundusImages')

# 2. Load and clean entire dataset
df = load_and_clean_data(CSV_PATH, IMG_DIR)

# 3. Map targets
df['classification_encoded'] = df['classification'].map(
    {'Emmetropia': 0, 'Myopia': 1, 'Hyperopia': 2}
)

# 4. Run Cross-Validation Pipeline
# (Splitting, Scaling, Training, and Evaluation are all handled internally per-fold)
results = run_cross_validation(
    df=df,
    model_name='densenet121',
    preprocess_input = preprocess_input,
    patient_col="ID",
    target_col="classification_encoded",
    n_splits=10,
    batch_size=16,
    epochs=30,
    learning_rate=0.0001,
    holdout_test_size=0.15,
)

Dataset Split: 866 samples for 10-Fold CV | 152 samples in Holdout Test Set (15%)

STARTING 10-FOLD CROSS VALIDATION


--- Fold 1/10 ---


I0000 00:00:1784993672.706539     260 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784993672.709396     260 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30
 1/96 ━━━━━━━━━━━━━━━━━━━━ 50:09 32s/step - accuracy: 0.2500 - loss: 1.8077

I0000 00:00:1784993709.971290     338 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 515ms/step - accuracy: 0.3916 - loss: 0.9625
Epoch 1: val_loss improved from None to 0.36702, saving model to best_densenet121_fold_1.h5

Epoch 1: finished saving model to best_densenet121_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 97s 692ms/step - accuracy: 0.4193 - loss: 0.7814 - val_accuracy: 0.6146 - val_loss: 0.3670
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.4985 - loss: 0.5465
Epoch 2: val_loss improved from 0.36702 to 0.34942, saving model to best_densenet121_fold_1.h5

Epoch 2: finished saving model to best_densenet121_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 47s 497ms/step - accuracy: 0.5156 - loss: 0.5360 - val_accuracy: 0.6562 - val_loss: 0.3494
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step - accuracy: 0.5604 - loss: 0.4992
Epoch 3: val_loss improved from 0.34942 to 0.34662, saving model to best_densenet121_fold_1.h5

Epoch 3: finished saving model to best_densenet121_fold_1.h5
96/96 ━━━━━━━━━━━━━━━━━━━━ 46s 480ms/step - accur